# Nemotron 3 Nano — RAFT / Reinforce-Rej on synthetic + train.csv (v20)

Rejection-sampling self-distillation, warm-started on the **0.85 SFT adapter**.
Same idea as v18 RAFT, but the prompt pool = **v19's synthetic generators
(infinite, perfectly verifiable) + real train.csv**, scored by v19's type-aware
verifier.

Loop: sample N rollouts/prompt (T=1.0) → verify with `\boxed{}` + per-type
checker → keep **mixed** groups' correct completions (Reinforce-Rej: drop
all-correct and all-wrong) → plain mean-NLL SFT on the kept (prompt, correct CoT)
pairs. Greedy-eval friendly; rarely regresses a strong base (unlike GRPO).

**Why this can beat plain SFT:** the model practices the exact rule families on
fresh instances it got *almost* right, distilling its own best reasoning. **Where
it plateaus:** instances with pass@N=0 yield no rollout → no signal. Raise
`N_ROLLOUTS`, seed easier synthetic first (curriculum), and oversample the weak
categories to push past 90.

In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

# ── run mode ──
TRAIN_ON_KAGGLE = 1
SEED            = 3407
MODEL_MAX_LEN   = 8192

# ── warm-start from the 0.85 SFT adapter (v18-exact resolution) ──
SFT_ADAPTER_MODEL_SLUG = ""    # "" -> skip kagglehub; use the dir below
SFT_ADAPTER_DIR        = "/kaggle/input/models/ramkan07/nemotron-lora-adaptor/pytorch/default/1"

# ── prompt pool ──
# "mixed": train.csv + synthetic (recommended).  "train_csv" / "synthetic" too.
DATA_SOURCE    = "mixed"
TRAIN_CSV_PATH = ("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
                  if TRAIN_ON_KAGGLE else "../data_generation/src/train.csv")
MAX_TRAIN_ROWS = 1500          # real rows per run
SYNTH_PER_CAT  = 400           # synthetic problems per category
POOL_CAP       = None          # cap total pool after shuffle (None = no cap)

# Prompt suffix = EXACTLY what the 0.85 SFT was trained with (match the policy's
# learned format so rollouts hit). No system prompt (eval feeds the bare problem).
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

# ── RAFT / Reinforce-Rej ──
SMOKE_TEST   = 1
RAFT_MODE    = "reinforce_rej"   # "raft" | "reinforce_rej"
TOP_K_KEEP   = 1                 # correct completions kept per mixed prompt
KEEP_FORMAT_VALID = True         # strong base emits <think>...</think>\boxed{} -> gate on it

N_PROMPTS    = 2500              # prompts sampled from the pool (real run)
N_ROLLOUTS   = 8                 # rollouts/prompt via num_return_sequences (8-16)
GEN_MAX_NEW  = 2048              # long enough to finish CoT + reach \boxed{}
GEN_TEMP     = 1.0
GEN_TOP_P    = 1.0

# ── RAFT-SFT scope (refine, don't smash the 0.85) ──
NUM_EPOCHS    = 1
TRAIN_MAX_LEN = 4096
RAFT_LR       = 3e-5

# ── smoke overrides ──
SMOKE_PROMPTS, SMOKE_ROLLOUTS, SMOKE_GEN_MAX_NEW, SMOKE_STEPS = 8, 4, 1024, 8
if SMOKE_TEST:
    N_PROMPTS, N_ROLLOUTS, GEN_MAX_NEW = SMOKE_PROMPTS, SMOKE_ROLLOUTS, SMOKE_GEN_MAX_NEW

MAX_LORA_RANK = 32
print({"SMOKE_TEST": SMOKE_TEST, "DATA_SOURCE": DATA_SOURCE, "RAFT_MODE": RAFT_MODE,
       "N_PROMPTS": N_PROMPTS, "N_ROLLOUTS": N_ROLLOUTS, "GEN_MAX_NEW": GEN_MAX_NEW,
       "KEEP_FORMAT_VALID": KEEP_FORMAT_VALID, "RAFT_LR": RAFT_LR})

In [ ]:
# Triton wheel — bundled in a Kaggle dataset (no internet at eval time).
if TRAIN_ON_KAGGLE:
    import glob, subprocess, site
    candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
    print("Triton wheels:", candidates)
    if candidates:
        target = "/kaggle/working/pydeps"
        os.makedirs(target, exist_ok=True)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
             "--upgrade", "--ignore-installed", candidates[0]],
            check=True,
        )
        if target not in sys.path:
            sys.path.insert(0, target)
        site.addsitedir(target)
    else:
        print("No Triton wheel found — assuming the env already provides Triton.")


In [ ]:
# ptxas for Blackwell (RTX 6000 Pro) — copy the bundled utility-script binary.
if TRAIN_ON_KAGGLE:
    import shutil, stat
    sys.path.insert(0, "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script")
    ptxas_src = ("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/"
                 "triton/backends/nvidia/bin/ptxas-blackwell")
    ptxas_dst = "/tmp/ptxas-blackwell"
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = ptxas_dst
        print("ptxas-blackwell ready ->", ptxas_dst)
    else:
        print("ptxas shim skipped (src missing or already present).")


In [ ]:
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
OUTPUT_ROOT      = "/kaggle/working/outputs/v20_raft" if TRAIN_ON_KAGGLE else "outputs/v20_raft"
ROLLOUT_CACHE    = os.path.join(OUTPUT_ROOT, "raft_rollouts.jsonl")
RAFT_ADAPTER_DIR = os.path.join(OUTPUT_ROOT, "raft_adapter")
TB_LOG_DIR       = os.path.join(OUTPUT_ROOT, "tb")
for d in (OUTPUT_ROOT, RAFT_ADAPTER_DIR, TB_LOG_DIR):
    os.makedirs(d, exist_ok=True)
print("Output root:", OUTPUT_ROOT)

In [ ]:
# Offline package install (EXACT v18 path) — find-links dir + mamba/causal wheels.
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )

    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MODEL_MAX_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # left padding for batched GENERATION (RAFT sampling); flipped to right for SFT later.
    tokenizer.padding_side = "left"
    print("Model loaded with Unsloth.")
else:
    print("TRAIN_ON_KAGGLE=0: skipping base model and tokenizer loading.")


## LoRA: warm-start from the 0.85 SFT adapter (v18-exact loadup)

In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
import re, os, glob

# ── Resolve the warm-start adapter dir (Kaggle MODEL slug -> dataset dir -> glob) ──
def _resolve_adapter_dir():
    if SFT_ADAPTER_MODEL_SLUG:
        try:
            import kagglehub
            p = kagglehub.model_download(SFT_ADAPTER_MODEL_SLUG)
            print(f"[adapter] kagglehub -> {p}")
            if os.path.exists(os.path.join(p, "adapter_config.json")):
                return p
            hits = glob.glob(os.path.join(p, "**", "adapter_config.json"), recursive=True)
            if hits:
                return os.path.dirname(sorted(hits, key=len)[0])
        except Exception as e:
            print(f"[adapter] kagglehub failed ({type(e).__name__}: {e}); trying dataset path")
    if os.path.exists(os.path.join(SFT_ADAPTER_DIR, "adapter_config.json")):
        return SFT_ADAPTER_DIR
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if hits:
        d = os.path.dirname(sorted(hits, key=len)[0])
        print(f"[adapter] auto-found -> {d}")
        return d
    return None

ADAPTER_DIR = _resolve_adapter_dir()

# Inventory linear modules (used only for the cold-start regex / debug).
linear_modules = []
for name, mod in model.named_modules():
    if mod.__class__.__name__ in ("Linear", "Linear4bit", "Linear8bitLt"):
        linear_modules.append(name)

target_regex = (
    r".*("
    r"self_attn\.(q|k|v|o)_proj"
    r"|mamba\.(in|out|x|dt|gate)_proj"
    r"|shared_experts\.(gate|up|down)_proj"
    r")$"
)

if ADAPTER_DIR and os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    print(f"[branch A: WARM-START] loading SFT adapter from {ADAPTER_DIR}")
    model = PeftModel.from_pretrained(model, ADAPTER_DIR, is_trainable=True)
else:
    print(f"[branch B: COLD-START] no adapter resolved "
          f"(slug={SFT_ADAPTER_MODEL_SLUG!r}, dir={SFT_ADAPTER_DIR!r}).")
    print(f"[branch B: COLD-START] fresh RSLoRA r=32 on base {BASE_MODEL_NAME}.")
    matched = [n for n in linear_modules if re.match(target_regex, n)]
    print(f"LoRA target regex matched {len(matched)} modules.")
    if len(matched) == 0:
        sample = [n for n in linear_modules if "expert" in n or "mamba" in n or "self_attn" in n][:20]
        raise RuntimeError(f"LoRA target_regex matched 0 modules. Sample: {sample}")
    lora_config = LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
        target_modules=target_regex, task_type=TaskType.CAUSAL_LM,
        use_rslora=True, use_dora=False,
    )
    model = get_peft_model(model, lora_config)

# ── grad-path hardening (real training vs dead zeros) ──
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
try:
    model.config.use_cache = False
except Exception:
    pass
model.print_trainable_parameters()

trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
n_train = sum(s for _, s in trainable)
print(f"[audit] {len(trainable)} trainable tensors  total={n_train/1e6:.1f}M")
assert 30_000_000 <= n_train <= 1_200_000_000, \
    f"[audit] trainable {n_train/1e6:.1f}M out of expected range -- inspect adapter / regex."

# ── submission-constraint check (rank <= 32) — non-fatal, just warn ──
try:
    _ranks = [getattr(c, "r", None) for c in model.peft_config.values()]
    _maxr = max([r for r in _ranks if r] or [0])
    print("adapter rank(s):", _ranks)
    if _maxr > MAX_LORA_RANK:
        print(f"  *** WARNING: rank {_maxr} > {MAX_LORA_RANK} -> submission WILL be rejected. ***")
except Exception as e:
    print("rank check skipped:", repr(e))


## Synthetic generators (vendored from `RLVR/synthetic_problems.py`)

In [ ]:
import random, string

CIPHER_SUBJECTS = ['alice','cat','rabbit','mouse','hatter','queen','king','knight',
    'princess','wizard','dragon','bird','turtle','student','teacher']
CIPHER_VERBS = ['follows','creates','draws','dreams','chases','reads','sees','watches',
    'explores','discovers','imagines','writes','studies','finds','found']
CIPHER_OBJECTS = ['castle','garden','book','mirror','key','map','door','forest','puzzle',
    'treasure','secret','tower','crystal','potion','message']
CIPHER_PLACES = ['wonderland','castle','forest','garden','valley','village','palace',
    'mountain','library','island','ocean','cave','school']
CIPHER_ADJECTIVES = ['the','magical','mysterious','golden','silver','bright','dark','clever',
    'colorful','hidden','ancient','strange','curious','wise']

def _random_phrase(rng, n_words=None):
    if n_words is None:
        n_words = rng.randint(2, 5)
    pool = CIPHER_SUBJECTS + CIPHER_VERBS + CIPHER_OBJECTS + CIPHER_PLACES + CIPHER_ADJECTIVES
    return " ".join(rng.choice(pool) for _ in range(n_words))

def _make_substitution_cipher(rng):
    letters = list(string.ascii_lowercase)
    shuffled = letters[:]
    rng.shuffle(shuffled)
    return dict(zip(letters, shuffled))

def _apply_cipher(text, cipher):
    return "".join(cipher.get(c, c) for c in text)

def generate_gravity_problem(rng, n_examples=5):
    g = round(rng.uniform(3.0, 30.0), 2)
    t_values = [round(rng.uniform(0.5, 6.0), 2) for _ in range(n_examples + 1)]
    examples, query_t = t_values[:n_examples], t_values[-1]
    lines = ["In Alice's Wonderland, the gravitational constant has been secretly changed. "
             "Here are some example observations:"]
    for t in examples:
        lines.append(f"For t = {t}s, distance = {round(0.5*g*t*t, 2)} m")
    lines.append(f"Now, determine the falling distance for t = {query_t}s given d = 0.5*g*t^2.")
    return "\n".join(lines), str(round(0.5*g*query_t*query_t, 2))

def generate_unit_conversion_problem(rng, n_examples=5):
    ratio = round(rng.uniform(0.3, 5.0), 4)
    inputs = [round(rng.uniform(1.0, 50.0), 2) for _ in range(n_examples + 1)]
    examples, query_inp = inputs[:n_examples], inputs[-1]
    lines = ["In Alice's Wonderland, a secret unit conversion is applied to measurements. For example:"]
    for inp in examples:
        lines.append(f"{inp} m becomes {round(inp*ratio, 2):.2f}")
    lines.append(f"Now, convert the following measurement: {query_inp} m")
    return "\n".join(lines), str(round(query_inp*ratio, 2))

_ROMAN = [(1000,"M"),(900,"CM"),(500,"D"),(400,"CD"),(100,"C"),(90,"XC"),
          (50,"L"),(40,"XL"),(10,"X"),(9,"IX"),(5,"V"),(4,"IV"),(1,"I")]
def _int_to_roman(n):
    out = []
    for val, sym in _ROMAN:
        while n >= val:
            out.append(sym); n -= val
    return "".join(out)

def generate_numeral_problem(rng, n_examples=4):
    used, nums = set(), []
    while len(nums) < n_examples:
        n = rng.randint(1, 3999)
        if n not in used:
            used.add(n); nums.append(n)
    q = rng.randint(1, 3999)
    while q in used:
        q = rng.randint(1, 3999)
    lines = ["In Alice's Wonderland, numbers are secretly converted into a different numeral system. "
             "Some examples are given below:"]
    for n in nums:
        lines.append(f"{n} -> {_int_to_roman(n)}")
    lines.append(f"Now, write the number {q} in the Wonderland numeral system.")
    return "\n".join(lines), _int_to_roman(q)

def generate_cipher_problem(rng, n_examples=5):
    cipher = _make_substitution_cipher(rng)
    phrases = [_random_phrase(rng, rng.randint(2, 5)) for _ in range(n_examples + 1)]
    examples, query = phrases[:n_examples], phrases[-1]
    lines = ["In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:"]
    for p in examples:
        lines.append(f"{_apply_cipher(p, cipher)} -> {p}")
    lines.append(f"Now, decrypt the following text: {_apply_cipher(query, cipher)}")
    return "\n".join(lines), query

def _random_8bit_function(rng):
    t = rng.choice(["xor_mask","rol","ror","not_rol","xor_two_rols","per_bit_gate"])
    if t == "xor_mask":
        mask = [rng.randint(0,1) for _ in range(8)]
        return lambda bits, m=mask: [b ^ mi for b, mi in zip(bits, m)]
    if t == "rol":
        k = rng.randint(1,7); return lambda bits, k=k: bits[k:] + bits[:k]
    if t == "ror":
        k = rng.randint(1,7); return lambda bits, k=k: bits[-k:] + bits[:-k]
    if t == "not_rol":
        k = rng.randint(1,7); return lambda bits, k=k: [1-b for b in bits[k:] + bits[:k]]
    if t == "xor_two_rols":
        k1 = rng.randint(0,7); k2 = rng.randint(0,7)
        while k2 == k1:
            k2 = rng.randint(0,7)
        def fn(bits, k1=k1, k2=k2):
            a = bits[k1:] + bits[:k1]; b = bits[k2:] + bits[:k2]
            return [x ^ y for x, y in zip(a, b)]
        return fn
    bit_maps = []
    for _ in range(8):
        idx = [rng.randint(0,7) for _ in range(rng.randint(1,3))]
        bit_maps.append((idx, rng.choice(["and","or","xor","nand","nor"])))
    def per_bit(bits, bm=bit_maps):
        res = []
        for idx, gate in bm:
            vals = [bits[i] for i in idx]
            if gate == "and":
                v = 1
                for x in vals: v &= x
            elif gate == "or":
                v = 0
                for x in vals: v |= x
            elif gate == "xor":
                v = 0
                for x in vals: v ^= x
            elif gate == "nand":
                v = 1
                for x in vals: v &= x
                v = 1 - v
            else:
                v = 0
                for x in vals: v |= x
                v = 1 - v
            res.append(v)
        return res
    return per_bit

def generate_bit_manipulation_problem(rng, n_examples=8):
    fn = _random_8bit_function(rng)
    rand_bits = lambda: [rng.randint(0,1) for _ in range(8)]
    inputs = [rand_bits() for _ in range(n_examples + 1)]
    examples, query = inputs[:n_examples], inputs[-1]
    lines = ["In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. "
             "The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, "
             "and possibly majority or choice functions.", "", "Here are some examples of input -> output:"]
    for b in examples:
        lines.append(f"{''.join(map(str,b))} -> {''.join(map(str,fn(b)))}")
    lines.append("")
    lines.append(f"Now, determine the output for: {''.join(map(str,query))}")
    return "\n".join(lines), "".join(map(str, fn(query)))

_SYM = list("!@#$%^&*()[]{}|<>?/\\'\"`~;:,.+-=")
def _random_symbol_string(rng, length):
    return "".join(rng.choice(_SYM) for _ in range(length))

def generate_equation_problem(rng, n_examples=4):
    chars = list(set(_SYM)); rng.shuffle(chars)
    cipher = {}
    half = len(chars) // 2
    for i, c in enumerate(chars[:half]):
        cipher[c] = chars[i + half]
    while len(cipher) < 10:
        c = rng.choice(_SYM)
        if c not in cipher:
            tgt = rng.choice(_SYM)
            if tgt not in cipher.values():
                cipher[c] = tgt
    apply = lambda s: "".join(cipher.get(c, c) for c in s)
    L = rng.randint(3, 6)
    inputs = [_random_symbol_string(rng, L) for _ in range(n_examples + 1)]
    query = inputs[-1]
    lines = ["In Alice's Wonderland, a secret set of transformation rules is applied to equations. "
             "Below are a few examples:"]
    for inp in inputs[:n_examples]:
        lines.append(f"{inp} = {apply(inp)}")
    lines.append(f"Now, determine the result for: {query}")
    return "\n".join(lines), apply(query)

GENERATORS = {
    "bit_manipulation": generate_bit_manipulation_problem,
    "cipher":           generate_cipher_problem,
    "numeral":          generate_numeral_problem,
    "unit_conversion":  generate_unit_conversion_problem,
    "gravity":          generate_gravity_problem,
    "equation":         generate_equation_problem,
}

def generate_synthetic_problems(n_per_category=500, seed=1337, categories=None):
    cats = categories or list(GENERATORS.keys())
    rng = random.Random(seed)
    out = []
    for cat in cats:
        gen = GENERATORS[cat]
        seen = set(); n = 0; attempts = 0
        while n < n_per_category and attempts < n_per_category * 6:
            attempts += 1
            try:
                prompt, answer = gen(rng)
            except Exception:
                continue
            if (prompt, answer) in seen:
                continue
            seen.add((prompt, answer))
            out.append({"prompt": prompt, "answer": answer, "category": cat, "source": "synthetic"})
            n += 1
        print(f"  {cat:18s} -> {n}")
    rng.shuffle(out)
    return out

# verifier-label map (category -> type used by verify_answer)
CATEGORY_TO_LABEL = {
    "bit_manipulation": "Bit Manipulation",
    "cipher":           "Text Encryption",
    "numeral":          "Number Base Conversion",
    "unit_conversion":  "Unit Conversion",
    "gravity":          "Gravitational Constant",
    "equation":         "Equation Transformation",
}
print("Generators ready:", list(GENERATORS.keys()))


## Verifiers + puzzle classifier (RLVR)

In [ ]:
import re

_BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

def _content(c):
    if isinstance(c, list) and c and isinstance(c[0], dict):
        return c[-1].get("content", "")
    return str(c)

def extract_boxed(text):
    idx = text.find("\\boxed{")
    if idx == -1:
        m = _BOXED_RE.search(text)
        return m.group(1).strip() if m else None
    depth, start = 1, idx + 7
    for i in range(start, len(text)):
        if text[i] == "{":   depth += 1
        elif text[i] == "}": depth -= 1
        if depth == 0:
            return text[start:i].strip()
    return text[start:].strip()

def _norm(s):
    return str(s).strip().lower().replace(" ", "").replace(",", "")

def _numeric_match(p, e, rel_tol=1e-2, abs_tol=1e-4):
    try:
        pf = float(str(p).replace(",", "").strip())
        ef = float(str(e).replace(",", "").strip())
    except (ValueError, TypeError):
        return False
    return abs(pf - ef) <= max(rel_tol * max(1.0, abs(ef)), abs_tol)

def _integer_match(p, e, bases=(2, 8, 10, 16)):
    ps, es = _norm(p), _norm(e)
    def parse(s, b):
        try:
            if b == 16 and s.startswith("0x"): s = s[2:]
            elif b == 2 and s.startswith("0b"): s = s[2:]
            return int(s, b)
        except ValueError:
            return None
    for b1 in bases:
        pv = parse(ps, b1)
        if pv is None:
            continue
        for b2 in bases:
            ev = parse(es, b2)
            if ev is not None and pv == ev:
                return True
    return False

def verify_answer(predicted, expected, label):
    if predicted is None:
        return False
    pn, en = _norm(predicted), _norm(expected)
    if pn == en:
        return True
    if label in ("Gravitational Constant", "Unit Conversion"):
        return _numeric_match(predicted, expected)
    if label in ("Bit Manipulation", "Number Base Conversion"):
        return _integer_match(predicted, expected)
    if label == "Text Encryption":
        return pn.replace("'", "") == en.replace("'", "")
    if label == "Equation Transformation":
        return _numeric_match(predicted, expected) or pn == en
    return _numeric_match(predicted, expected)

# ── puzzle-type classifier (for train.csv rows; order matters) ────────────────
_LABEL_PATTERNS = [
    ("Gravitational Constant",  re.compile(r"gravit|falling distance|free.?fall|0\.5\s*\*?\s*g\s*\*?\s*t", re.I)),
    ("Number Base Conversion",  re.compile(r"numeral system|number.*convert|base[- ]?\d|radix", re.I)),
    ("Unit Conversion",         re.compile(r"unit conversion|m becomes|measurement", re.I)),
    ("Text Encryption",         re.compile(r"encrypt|decrypt|cipher", re.I)),
    ("Bit Manipulation",        re.compile(r"bit manipulation|8.?bit|binary number|bitwise", re.I)),
    ("Equation Transformation", re.compile(r"transformation rule|set of transformation|determine the result", re.I)),
]
def classify_label(prompt):
    for lab, pat in _LABEL_PATTERNS:
        if pat.search(prompt or ""):
            return lab
    return "Unknown"

# smoke tests
assert verify_answer("9.81", "9.80", "Gravitational Constant")
assert verify_answer("00101010", "42", "Bit Manipulation")
assert verify_answer("HELLO", "hello", "Text Encryption")
assert not verify_answer("9.81", "10.0", "Gravitational Constant")
print("Verifier smoke tests pass.")


## Build the prompt pool (train.csv + synthetic)

Each item = `{raw_prompt, answer, label}`. `label` is the known category for
synthetic, else `classify_label` for train.csv rows -> picks the verifier. The
model never sees the label.

In [ ]:
import pandas as pd, random as _r

pool = []
if DATA_SOURCE in ("train_csv", "mixed"):
    df = pd.read_csv(TRAIN_CSV_PATH)
    if not {"prompt", "answer"}.issubset(df.columns):
        raise ValueError(f"train.csv needs prompt+answer; got {list(df.columns)}")
    df = df.dropna(subset=["prompt", "answer"])
    if MAX_TRAIN_ROWS:
        df = df.sample(min(MAX_TRAIN_ROWS, len(df)), random_state=SEED)
    for r in df.itertuples(index=False):
        raw = str(r.prompt).replace("\r\n", "\n")
        pool.append({"raw_prompt": raw, "answer": str(r.answer),
                     "label": classify_label(raw), "source": "train_csv"})
    print(f"train.csv rows: {len(pool)}")
if DATA_SOURCE in ("synthetic", "mixed"):
    syn = generate_synthetic_problems(n_per_category=SYNTH_PER_CAT, seed=SEED)
    for s in syn:
        pool.append({"raw_prompt": s["prompt"], "answer": s["answer"],
                     "label": CATEGORY_TO_LABEL[s["category"]], "source": "synthetic"})
    print(f"+ synthetic -> total {len(pool)}")

# keep only answers that round-trip through the boxed extractor (verifier parity)
def _roundtrips(a):
    return extract_boxed("\\boxed{" + str(a) + "}") == str(a)
pool = [e for e in pool if _roundtrips(e["answer"])]

_r.Random(SEED).shuffle(pool)
if POOL_CAP:
    pool = pool[:POOL_CAP]
prompts = pool[:N_PROMPTS]

from collections import Counter
print(f"Pool kept {len(pool)}; sampling {len(prompts)}.  label dist:",
      dict(Counter(p["label"] for p in prompts)))

## Sample N rollouts/prompt, verify, cache

Per-prompt generation (NO cross-prompt padding — Nemotron Mamba-2 NaNs on
left-pad batches). N rollouts via `num_return_sequences`. Scored by `verify_answer`
with the per-prompt `label`. Cached to disk so an SFT crash never costs sampling.

In [ ]:
import json, time, gc, torch
from transformers import LogitsProcessor, LogitsProcessorList

def build_gen_prompt(raw):
    msgs = [{"role": "user", "content": raw + PROMPT_SUFFIX}]
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                                             add_generation_prompt=True, enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

class SanitizeLogits(LogitsProcessor):
    def __init__(self): self.hits = 0
    def __call__(self, input_ids, scores):
        if not torch.isfinite(scores).all():
            self.hits += 1
            scores = torch.nan_to_num(scores, nan=-1e4, posinf=1e4, neginf=-1e4)
        return scores

def _set_gen_mode(on):
    try:
        if on: model.gradient_checkpointing_disable()
        else:  model.gradient_checkpointing_enable()
    except Exception as e: print("grad-ckpt toggle warn:", e)
    try: model.config.use_cache = on
    except Exception:
        try: model.base_model.config.use_cache = on
        except Exception: pass

if os.path.exists(ROLLOUT_CACHE) and not SMOKE_TEST:
    print(f"Rollout cache exists -> reusing {ROLLOUT_CACHE} (delete to re-sample).")
else:
    if os.path.exists(ROLLOUT_CACHE): os.remove(ROLLOUT_CACHE)
    _set_gen_mode(True); model.eval()
    tokenizer.padding_side = "left"
    guard = SanitizeLogits(); lp = LogitsProcessorList([guard])
    torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time(); n_done = n_corr = 0
    rep = max(1, len(prompts) // 20)
    fout = open(ROLLOUT_CACHE, "w", encoding="utf-8")
    try:
        for ex in prompts:
            enc = tokenizer(build_gen_prompt(ex["raw_prompt"]), return_tensors="pt",
                            truncation=True, max_length=TRAIN_MAX_LEN).to(model.device)
            plen = enc["input_ids"].shape[1]
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=GEN_MAX_NEW, do_sample=True,
                                     temperature=GEN_TEMP, top_p=GEN_TOP_P,
                                     num_return_sequences=N_ROLLOUTS,
                                     pad_token_id=tokenizer.pad_token_id, logits_processor=lp)
            comps, flags = [], []
            for seq in out:
                gen = tokenizer.decode(seq[plen:], skip_special_tokens=True)
                ok = bool(verify_answer(extract_boxed(gen), ex["answer"], ex["label"]))
                comps.append(gen); flags.append(ok); n_corr += int(ok)
            if n_done == 0:
                print("[rollout-dbg] rollout0 tail:", repr(comps[0][-200:]))
                print("[rollout-dbg] pred=", repr(extract_boxed(comps[0])), "gold=", repr(ex["answer"]),
                      "label=", ex["label"], "ok=", flags[0])
            fout.write(json.dumps({"raw_prompt": ex["raw_prompt"], "answer": ex["answer"],
                                   "label": ex["label"], "completions": comps, "correct": flags}) + "\n")
            n_done += 1
            del out, enc; torch.cuda.empty_cache()
            if n_done % rep == 0 or n_done == len(prompts):
                el = time.time() - t0
                print(f"  [{n_done}/{len(prompts)}] {el/60:.1f}min ~{el/max(1,n_done):.1f}s/p "
                      f"corr={n_corr} NaN-guard={guard.hits} "
                      f"peakVRAM={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    finally:
        fout.close(); _set_gen_mode(False); model.train()
    el = time.time() - t0
    print(f"\nSampling done: {n_done} x {N_ROLLOUTS} in {el/60:.1f}min  "
          f"hit-rate {n_corr}/{n_done*N_ROLLOUTS} ({100*n_corr/max(1,n_done*N_ROLLOUTS):.1f}%)")
    print(f"NaN-guard hits: {guard.hits} (MUST be 0)")
    if n_corr == 0:
        print("[WARN] 0 correct rollouts -> warm-start not loaded, or GEN_MAX_NEW too short.")

## Filter rollouts -> SFT corpus (Reinforce-Rej)

`reinforce_rej` keeps correct completions only from **mixed** groups (drops
all-correct p=1 and all-wrong p=0). Fallbacks widen if the corpus is empty.

In [ ]:
import json as _json
from datasets import Dataset as HFDataset

groups = []
with open(ROLLOUT_CACHE, encoding="utf-8") as f:
    for line in f:
        groups.append(_json.loads(line))

def _has_think_before_boxed(t):
    return ("<think>" in t and "</think>" in t and "\\boxed{" in t
            and t.find("</think>") < t.rfind("\\boxed{"))

def build_corpus(mode, keep_format):
    n_wrong = n_all = n_mix = 0; out = []
    for g in groups:
        flags = g["correct"]; n_ok = sum(flags)
        p = n_ok / max(1, len(flags))
        if n_ok == 0:
            n_wrong += 1; continue
        if p >= 1.0:
            n_all += 1
            if mode == "reinforce_rej": continue
        else:
            n_mix += 1
        cand = [c for c, ok in zip(g["completions"], flags) if ok]
        if keep_format:
            fmt = [c for c in cand if _has_think_before_boxed(c)]
            cand = fmt if fmt else cand
        cand = sorted(cand, key=len)[:TOP_K_KEEP]   # shortest clean CoT
        for comp in cand:
            out.append({"user": g["raw_prompt"] + PROMPT_SUFFIX, "assistant": comp.strip()})
    return out, dict(all_wrong=n_wrong, all_correct=n_all, mixed=n_mix)

records, stats = build_corpus(RAFT_MODE, KEEP_FORMAT_VALID)
print(f"[try1] {RAFT_MODE} fmt={KEEP_FORMAT_VALID} -> corpus={len(records)} {stats}")
if len(records) == 0 and KEEP_FORMAT_VALID:
    records, stats = build_corpus(RAFT_MODE, False)
    print(f"[try2] {RAFT_MODE} fmt=False -> corpus={len(records)} {stats}")
if len(records) == 0 and RAFT_MODE == "reinforce_rej":
    records, stats = build_corpus("raft", False)
    print(f"[try3] raft fmt=False -> corpus={len(records)} {stats}")
if len(records) == 0:
    raise RuntimeError("Empty corpus: 0 correct rollouts. Raise N_ROLLOUTS, confirm "
                       "warm-start loaded, or seed easier synthetic. Delete the cache to re-sample.")

raw_ds = HFDataset.from_list(records)
print(f"\nFinal corpus: {len(records)} examples")
print("--- sample assistant target tail ---\n", records[0]["assistant"][-200:])

In [ ]:
def tokenize_with_assistant_mask(example):
    full_msgs   = [{"role": "user", "content": example["user"]},
                   {"role": "assistant", "content": example["assistant"]}]
    prefix_msgs = [{"role": "user", "content": example["user"]}]
    def render(msgs, add_gen):
        try:
            return tokenizer.apply_chat_template(msgs, tokenize=False,
                                                 add_generation_prompt=add_gen, enable_thinking=True)
        except TypeError:
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=add_gen)
    full_ids   = tokenizer(render(full_msgs, False), add_special_tokens=False,
                           truncation=True, max_length=TRAIN_MAX_LEN)["input_ids"]
    prefix_ids = tokenizer(render(prefix_msgs, True), add_special_tokens=False)["input_ids"]
    cutoff = min(len(prefix_ids), len(full_ids))
    labels = list(full_ids)
    for i in range(cutoff): labels[i] = -100
    return {"input_ids": full_ids, "labels": labels}

tokenized_ds = raw_ds.map(tokenize_with_assistant_mask,
                          remove_columns=raw_ds.column_names, desc="Tokenize + mask")
before = len(tokenized_ds)
tokenized_ds = tokenized_ds.filter(lambda ex: any(t != -100 for t in ex["labels"]))
print(f"Kept {len(tokenized_ds)}/{before} rows with >=1 unmasked assistant token.")
if len(tokenized_ds) == 0:
    raise RuntimeError("tokenized_ds EMPTY after masking -> rollouts lack a closing </think>. "
                       "Set KEEP_FORMAT_VALID=True or re-sample with a stronger base.")
_ex0 = tokenized_ds[0]
_un = [t for t, l in zip(_ex0["input_ids"], _ex0["labels"]) if l != -100]
print(f"[mask-check] row0 total={len(_ex0['input_ids'])} unmasked={len(_un)}")
print("[mask-check] decoded unmasked:", repr(tokenizer.decode(_un)[:200]))
import numpy as np
_lens = np.array([len(x) for x in tokenized_ds["input_ids"]])
print(f"len min={_lens.min()} mean={_lens.mean():.0f} p90={int(np.percentile(_lens,90))} max={_lens.max()}")

In [ ]:
import torch
class CompletionOnlyDataCollator:
    def __init__(self, tokenizer, label_pad_id=-100):
        self.pad_id = tokenizer.pad_token_id; self.label_pad_id = label_pad_id
    def __call__(self, features):
        maxlen = max(len(f["input_ids"]) for f in features)
        ii, ll, am = [], [], []
        for f in features:
            ids = list(f["input_ids"]); lab = list(f["labels"]); pad = maxlen - len(ids)
            ii.append(ids + [self.pad_id]*pad); ll.append(lab + [self.label_pad_id]*pad)
            am.append([1]*len(ids) + [0]*pad)
        return {"input_ids": torch.tensor(ii), "attention_mask": torch.tensor(am),
                "labels": torch.tensor(ll)}
tokenizer.padding_side = "right"   # SFT loss wants right padding
data_collator = CompletionOnlyDataCollator(tokenizer)
print("Collator ready.")

## Vanilla `transformers.Trainer` (NOT trl.SFTTrainer)

SFTTrainer's dataset-prep can drop the masked labels -> empty mask -> silent 0.0.
Plain Trainer passes input_ids/labels to compute_loss verbatim; fp32 reduction,
non-finite logits sanitised+recomputed, loud counters.

In [ ]:
import os, sys
os.environ["TORCHDYNAMO_DISABLE"] = "1"; os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch, torch._dynamo
import torch.nn.functional as F
torch._dynamo.config.disable = True; torch._dynamo.reset()
for _m in list(sys.modules):
    if _m == "trl" or _m.startswith("trl.") or "unsloth" in _m.lower():
        del sys.modules[_m]
sys.meta_path = [f for f in sys.meta_path if "unsloth" not in type(f).__module__.lower()]
from transformers import Trainer, TrainingArguments

class RaftTrainer(Trainer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k); self._c = {"nonfinite": 0, "dead": 0, "dbg": 0}
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); outputs = model(**inputs); logits = outputs.logits
        sl = logits[:, :-1, :]; lab = labels[:, 1:].to(sl.device); B, T, V = sl.shape
        nll = F.cross_entropy(sl.reshape(-1, V), lab.reshape(-1),
                              ignore_index=-100, reduction="none").view(B, T)
        if not torch.isfinite(nll).all():
            self._c["nonfinite"] += 1
            sl2 = torch.nan_to_num(sl, nan=0.0, posinf=30.0, neginf=-30.0)
            nll = F.cross_entropy(sl2.reshape(-1, V), lab.reshape(-1),
                                  ignore_index=-100, reduction="none").view(B, T)
            nll = torch.nan_to_num(nll, nan=0.0, posinf=30.0, neginf=0.0)
        mask = (lab != -100); loss = nll.float()[mask].mean()
        if self._c["dbg"] < 3:
            self._c["dbg"] += 1
            print(f"[loss-dbg step~{self.state.global_step}] loss={float(loss):.4f} "
                  f"unmasked={int(mask.sum())} nonfinite={self._c['nonfinite']}")
        if not torch.isfinite(loss):
            self._c["dead"] += 1
            if self._c["dead"] <= 5:
                print(f"[loss-WARN] non-finite loss step {self.state.global_step}; zeroing.")
            loss = (logits.float().sum() * 0.0).requires_grad_(True)
        return (loss, outputs) if return_outputs else loss
print("RaftTrainer defined.")

In [ ]:
_max_steps = SMOKE_STEPS if SMOKE_TEST else -1
args = TrainingArguments(
    output_dir                   = os.path.join(OUTPUT_ROOT, "raft_run"),
    num_train_epochs             = NUM_EPOCHS,
    max_steps                    = _max_steps,
    per_device_train_batch_size  = 1,
    gradient_accumulation_steps  = 8,
    learning_rate                = RAFT_LR,
    lr_scheduler_type            = "cosine",
    warmup_ratio                 = 0.05,
    weight_decay                 = 0.01,
    max_grad_norm                = 1.0,
    optim                        = "paged_adamw_8bit",
    bf16                         = True,
    gradient_checkpointing       = False,   # model already checkpoints (adapter cell)
    remove_unused_columns        = False,
    logging_steps                = 1 if SMOKE_TEST else 5,
    logging_dir                  = TB_LOG_DIR,
    report_to                    = "none",
    save_strategy                = "no",    # disk-safe: ~4GB adapter, no mid-run ckpts
    seed                         = SEED,
    dataloader_num_workers       = 2,
)
print(f"Args ready. mode={'SMOKE' if SMOKE_TEST else 'REAL'} LR={RAFT_LR} "
      f"eff_batch={args.per_device_train_batch_size*args.gradient_accumulation_steps}")

## Launch RAFT SFT + save

In [ ]:
import gc, time, torch, glob, shutil

trainer = RaftTrainer(model=model, args=args, train_dataset=tokenized_ds,
                      data_collator=data_collator)
torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
t0 = time.time(); train_err = None
try:
    trainer.train(); print(f"RAFT SFT done in {(time.time()-t0)/60:.1f} min")
except Exception as e:
    train_err = e; print(f"[TRAIN ERROR] {type(e).__name__}: {e}")
print(f"PEAK VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
print(f"[loss-counters] nonfinite={trainer._c['nonfinite']} dead={trainer._c['dead']}")

# free disk before saving the ~4GB adapter
_run = os.path.join(OUTPUT_ROOT, "raft_run")
if os.path.isdir(_run): shutil.rmtree(_run, ignore_errors=True)

os.makedirs(RAFT_ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(RAFT_ADAPTER_DIR); tokenizer.save_pretrained(RAFT_ADAPTER_DIR)
have = {f: os.path.exists(os.path.join(RAFT_ADAPTER_DIR, f))
        for f in ("adapter_config.json", "adapter_model.safetensors")}
print("Adapter saved ->", RAFT_ADAPTER_DIR, have)
if SMOKE_TEST:
    print("\n[SMOKE] green if hit-rate>0, NaN-guard=0, loss-dbg real >0, dead=0.")
if train_err is not None: raise train_err

## Package submission.zip (eval-gate first!)

RAFT can still regress a strong SFT on some categories. **Only submit if it beats
the 0.85 on a held-out slice** — else keep the SFT adapter (best-of-2).

In [ ]:
import json, zipfile, os
needed = ["adapter_config.json", "adapter_model.safetensors"]
src_dir = RAFT_ADAPTER_DIR
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else OUTPUT_ROOT
missing = [n for n in needed if not os.path.exists(os.path.join(src_dir, n))]
if missing:
    raise FileNotFoundError(f"Adapter files {missing} missing in {src_dir}; re-run launch.")
cfg_path = os.path.join(src_dir, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True; cfg["lora_dropout"] = 0.0
assert cfg.get("r", MAX_LORA_RANK) <= MAX_LORA_RANK, "rank > 32 -> rejected"
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)
zip_path = os.path.join(WORKING, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in needed: zf.write(os.path.join(src_dir, fn), fn)
print(f"submission.zip -> {zip_path}  ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

---
**Run order**
1. `SFT_ADAPTER_DIR` = your 0.85 adapter. Confirm the LoRA cell prints `[branch A: WARM-START]` + rank ≤ 32.
2. `SMOKE_TEST=1`, run all. Check: `[rollout-dbg] ok=True`, `hit-rate > 0`, `NaN-guard hits: 0`, corpus non-empty, `loss-dbg` real >0.
3. `SMOKE_TEST=0`, delete `raft_rollouts.jsonl`, Save & Run All.
4. **Eval-gate** vs the 0.85 before submitting.

**Push past 90 (when this plateaus):**
- Raise `N_ROLLOUTS` to 16 on the residual; **curriculum** = easy synthetic first.
- **Oversample the weak categories** (per-`label` hit-rate from sampling tells you which).
- 2–3 RAFT rounds: re-upload this adapter as `SFT_ADAPTER_DIR`, delete the cache, re-run — solved-last-round prompts drop out, harder ones become mixed.
- Add the v19 **SDPG** dense reward only for the stubborn residual where pass@N stays low.